# ENS 2016–2017 — Blood Lead and Cardiometabolic Markers

This notebook implements the analyses examining the association between blood lead concentration and cardiometabolic markers among participants in the 2016–2017 Chilean National Health Survey.

## Data Processing and Model Specifications

### Model Covariates

- Smoking: four categories derived from `ta3`.
- Alcohol consumption: an indicator of consumption during the past 12 months derived from `m7p2`.
- Physical activity: an indicator derived from `a17`.
- Education: years of schooling treated as a continuous variable.

### Descriptive Categories in Table 1

Descriptive categories are constructed separately from the model covariates:

- Smoking: current smoker, former smoker, and never smoker.
- Alcohol consumption: never drinker/abstainer, former drinker, and current drinker.
- Physical activity according to GPAQ: low, moderate, and high levels.

### Handling of Special Values

- The code `-7777` is treated as a missing value for Pb concentration.
- For the descriptive statistics in Table 1, HbA1c values recorded as `>14.7` are represented by the reported limit of `14.7`.
- In the regression models, these records are not treated as exact numerical measurements and are excluded through complete-case analysis.

## Results Verification

The notebook compares the main blood pressure results with reference values from the manuscript. This comparison is performed after model fitting and does not modify the estimates.

Spline, interaction, and metal mixture analyses are calculated from the data and should be assessed alongside their assumptions and limitations.

Full reproduction requires the augmented dataset used in the study and execution of all cells in order. Any discrepancies should be investigated by reviewing the data, variable coding, analytical sample, and software versions.



## Analytical Workflow and Statistical Methods

The analysis follows these steps:

1. Load the augmented dataset.
2. Clean and recode variables.
3. Construct complete-case samples for each model.
4. Fit the statistical models.
5. Calculate estimates, 95% confidence intervals, and p-values.
6. Compare selected blood pressure results with manuscript reference values.

Reference values are used only for post-estimation checks. They are not included in model fitting and do not replace calculated results. Any discrepancies are retained for review.

### Analytical Specifications

- **Primary exposure:** `log2(Pb)`, calculated from `Plomo_µgdL`. The coefficient represents the change in the outcome associated with a doubling of blood lead concentration.
- **Adjustment covariates:** age, BMI, geographic area (`Zona`), sex, smoking (`ta3`, four categories), alcohol consumption during the past 12 months (`m7p2`), years of schooling, and physical activity (`a17`).
- **Blood pressure outcomes:** systolic and diastolic blood pressure (SBP and DBP; `PAS` and `PAD` in the code), calculated as the mean of the available readings across three measurements.
- **Primary weighted models:** weighted least squares (WLS) using `Fexp_F1F2EX2p_Corr`, with stratum fixed effects (`Estrato`) and cluster-robust covariance by primary sampling unit (`Conglomerado`).
- **Robustness analyses:** unweighted ordinary least squares with HC3 robust standard errors and unweighted median regression.
- **Spline analyses:** fully adjusted, unweighted models comparing a linear exposure term with a cubic B-spline basis specified with three degrees of freedom.
- **Interaction analyses:** weighted models using the main adjustment covariates, stratum fixed effects, and cluster-robust covariance. Interaction terms are assessed using joint Wald tests.
- **Exploratory mixture analysis:** a Python implementation of linear quantile g-computation using quantile-based exposure categories, weighted least squares, and a joint effect (ψ) calculated as the sum of the coefficients for the categorized exposures. Uncertainty is evaluated using 200 individual-level bootstrap resamples with seed 42. Tied exposure values may reduce the effective number of categories, which is reported for each metal. This bootstrap does not resample survey strata or primary sampling units.


In [1]:
# ============================================================
# 0. INSTALLATION AND IMPORTS
# ============================================================
%pip -q install pyreadstat openpyxl statsmodels patsy scipy

import os, re, math, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.stats as st
import statsmodels.api as sm
import statsmodels.formula.api as smf

from patsy import dmatrix
from statsmodels.stats.multitest import multipletests
from IPython.display import display
from google.colab import files

warnings.filterwarnings("ignore")
np.random.seed(42)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)

print("Environment ready")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 14.8 MB/s eta 0:00:00
Environment ready


In [2]:
# 1. Load the augmented dataset

from pathlib import Path
import shutil
import tempfile
import zipfile

import pyreadstat
from google.colab import files

print("Upload the augmented .sav dataset or a ZIP containing a single .sav file:")
uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one .sav or ZIP file.")

input_path = Path(next(iter(uploaded)))

with tempfile.TemporaryDirectory() as temp_dir:
    if input_path.suffix.lower() == ".sav":
        sav_path = input_path

    elif input_path.suffix.lower() == ".zip":
        with zipfile.ZipFile(input_path) as archive:
            candidates = [
                entry for entry in archive.infolist()
                if not entry.is_dir()
                and entry.filename.lower().endswith(".sav")
            ]

            if len(candidates) != 1:
                raise ValueError(
                    "The ZIP must contain exactly one .sav file. "
                    f"Found {len(candidates)}."
                )

            sav_path = Path(temp_dir) / "dataset.sav"
            with archive.open(candidates[0]) as source:
                with sav_path.open("wb") as destination:
                    shutil.copyfileobj(source, destination)

    else:
        raise ValueError("Unsupported format. Upload a .sav or ZIP file.")

    raw, meta = pyreadstat.read_sav(
        sav_path,
        apply_value_formats=False,
        user_missing=False,
    )

print(f"Dataset loaded: {raw.shape[0]:,} rows and {raw.shape[1]:,} columns.")

if len(raw) != 6233:
    print(
        "Note: the manuscript reports an initial sample of 6,233 participants; "
        f"this dataset contains {len(raw):,} rows."
    )

Upload the augmented .sav dataset or a ZIP containing a single .sav file:


Saving Base_de_datos_Encuesta_Nacional_de_Salud_2016-2017(ENS)_Formulario_1_2_EX_MINSAL_EPI_CIDI_SF_Comuna_metales_pesados (1).zip to Base_de_datos_Encuesta_Nacional_de_Salud_2016-2017(ENS)_Formulario_1_2_EX_MINSAL_EPI_CIDI_SF_Comuna_metales_pesados (1).zip
Dataset loaded: 6,233 rows and 1,165 columns.


In [3]:
# 2. Check required variables

required = [
    "Plomo_µgdL", "Arsenico_µgL", "Cadmio_µgL", "Mercurio_µgL",
    "Fexp_F1F2EX2p_Corr", "Estrato", "Conglomerado",
    "Edad", "Sexo", "IMC", "Zona", "Region",
    "ta3", "m7p2", "m7p5", "GPAQ", "a17", "anos_estudio_MINSAL_1",
    "Colesterol_Total", "Colesterol_HDL", "Trigliceridos",
    "Glucosa", "Hemoglobina_A1C",
    "m2p8_1", "m2p9_1", "m2p10_1",
    "m2p8_2", "m2p9_2", "m2p10_2",
]

missing = [column for column in required if column not in raw.columns]

if missing:
    raise KeyError(
        "The dataset is missing required variables: " + ", ".join(missing)
    )

print("All required columns are present.")

# Identify a candidate variable for antihypertensive medication use.
# Confirm its meaning and coding against the SPSS labels or ENS codebook.
antihyp_candidates = [
    "m2p1", "M2P1", "HTA_tratamiento", "antihipertensivos"
]

ANTIHYP = next(
    (column for column in antihyp_candidates if column in raw.columns),
    None,
)

if ANTIHYP is None:
    print(
        "No antihypertensive medication variable was found among the "
        "candidate names. Review the SPSS labels and set ANTIHYP manually "
        "if an appropriate variable is available. Otherwise, the medication "
        "sensitivity analysis will be skipped."
    )
else:
    print(f"Candidate antihypertensive medication variable: {ANTIHYP}")
    print("Verify that its coding is 1 = yes and 2 = no before proceeding.")


All required columns are present.
Candidate antihypertensive medication variable: m2p1
Verify that its coding is 1 = yes and 2 = no before proceeding.


In [4]:
# 3. Data cleaning and derived variables

d0 = raw.copy()

SPECIAL = [-9999, -8888, -7777, 9999, 8888, 999, 888]

# HbA1c: use the reported limit for descriptives and exclude
# non-exact measurements from regression models.
d0["HbA1c_raw"] = d0["Hemoglobina_A1C"].astype(str).str.strip()

d0["HbA1c_desc"] = pd.to_numeric(
    d0["HbA1c_raw"].replace({
        ">14.7": "14.7",
        "nan": np.nan,
        "": np.nan,
    }),
    errors="coerce",
)

d0["HbA1c_model"] = pd.to_numeric(
    d0["HbA1c_raw"],
    errors="coerce",
)

numeric_cols = [
    "Plomo_µgdL", "Arsenico_µgL", "Cadmio_µgL", "Mercurio_µgL",
    "Fexp_F1F2EX2p_Corr", "Estrato", "Conglomerado",
    "Edad", "Sexo", "IMC", "Zona", "Region",
    "ta3", "m7p2", "m7p5", "GPAQ", "a17", "anos_estudio_MINSAL_1",
    "Colesterol_Total", "Colesterol_HDL", "Trigliceridos", "Glucosa",
    "m2p8_1", "m2p9_1", "m2p10_1",
    "m2p8_2", "m2p9_2", "m2p10_2",
]

if ANTIHYP is not None:
    numeric_cols.append(ANTIHYP)

for c in numeric_cols:
    d0[c] = pd.to_numeric(d0[c], errors="coerce")
    d0.loc[d0[c].isin(SPECIAL), c] = np.nan

for c in ["HbA1c_desc", "HbA1c_model"]:
    d0.loc[d0[c].isin(SPECIAL), c] = np.nan

# Blood pressure: mean of available readings.
d0["PAS"] = d0[["m2p8_1", "m2p9_1", "m2p10_1"]].mean(axis=1)
d0["PAD"] = d0[["m2p8_2", "m2p9_2", "m2p10_2"]].mean(axis=1)

# Descriptive categories for Table 1.
# Internal labels are retained for compatibility with subsequent cells.
d0["tabaco_cat_desc"] = np.select(
    [
        d0["ta3"].isin([1, 2]),
        d0["ta3"].eq(3),
        d0["ta3"].eq(4),
    ],
    ["Fumador actual", "Exfumador", "Nunca fumó"],
    default=None,
)

d0["alcohol_cat_desc"] = np.select(
    [
        d0["m7p2"].eq(2),
        d0["m7p2"].eq(1) & d0["m7p5"].eq(2),
        d0["m7p2"].eq(1) & d0["m7p5"].eq(1),
    ],
    ["Nunca/abstemio", "Exbebedor", "Bebedor actual"],
    default=None,
)

d0["af_cat_desc"] = d0["GPAQ"].map({
    1: "Bajo",
    2: "Medio",
    3: "Alto",
})

d0["educ_cat"] = pd.cut(
    d0["anos_estudio_MINSAL_1"],
    bins=[-np.inf, 7, 12, np.inf],
    labels=["<8 años", "8–12 años", "≥13 años"],
)

# Model covariates.
d0["smoking"] = d0["ta3"].map({
    1: "daily",
    2: "occasional",
    3: "former",
    4: "never",
})

d0["alcohol12m"] = d0["m7p2"].map({1: 1, 2: 0})
d0["edu_years"] = d0["anos_estudio_MINSAL_1"]
d0["phys_active"] = d0["a17"].map({1: 1, 2: 1, 3: 1, 4: 0})

# Compute log2 only for positive Pb concentrations.
d0["log2Pb"] = np.log2(
    d0["Plomo_µgdL"].where(d0["Plomo_µgdL"] > 0)
)

d0["obesidad"] = np.where(
    d0["IMC"].isna(),
    np.nan,
    (d0["IMC"] >= 30).astype(int),
)

# Geographic macrozones for exploratory analyses.
def macrozona(region):
    if pd.isna(region):
        return np.nan

    r = int(region)

    if r in [1, 2, 15]:
        return "Norte"
    if r in [3, 4, 5]:
        return "Centro-norte"
    if r == 13:
        return "Metropolitana"
    if r in [6, 7, 8, 16]:
        return "Centro-sur"
    if r in [9, 10, 11, 12, 14]:
        return "Sur"

    return np.nan


d0["macrozona"] = d0["Region"].apply(macrozona)

# Antihypertensive medication indicator.
if ANTIHYP is not None:
    d0["antihip"] = d0[ANTIHYP].map({1: 1, 2: 0})
else:
    d0["antihip"] = np.nan

# Valid survey design: positive weight and nonmissing stratum/PSU.
design_ok = (
    d0["Fexp_F1F2EX2p_Corr"].notna()
    & (d0["Fexp_F1F2EX2p_Corr"] > 0)
    & d0["Estrato"].notna()
    & d0["Conglomerado"].notna()
)

# Descriptive sample and adult sample for subsequent model fitting.
desc = d0.loc[design_ok].copy()
d = d0.loc[d0["Edad"] >= 18].copy()

# Data checks: counts and unweighted Pb quantiles.
pb_available = d0["Plomo_µgdL"].notna()
pb_design_ok = pb_available & design_ok

pb_quantiles = (
    d0.loc[pb_design_ok, "Plomo_µgdL"]
    .quantile([0.25, 0.50, 0.75])
    .to_dict()
)

print("Total N:", len(d0))
print("N with valid survey design:", len(desc))
print("N with available Pb:", int(pb_available.sum()))
print("N with available Pb and valid survey design:", int(pb_design_ok.sum()))
print(
    "N with Pb = 1.0 µg/dL and valid survey design:",
    int((d0["Plomo_µgdL"].eq(1.0) & design_ok).sum()),
)
print("Pb P25/P50/P75 (unweighted):", pb_quantiles)

Total N: 6233
N with valid survey design: 3847
N with available Pb: 3614
N with available Pb and valid survey design: 3600
N with Pb = 1.0 µg/dL and valid survey design: 2485
Pb P25/P50/P75 (unweighted): {0.25: 1.0, 0.5: 1.0, 0.75: 1.15}


## 4. Estimation Functions

- **Weighted means and proportions** use the sampling weight `Fexp_F1F2EX2p_Corr`.
- **Primary SBP and DBP models** use weighted least squares (WLS), with stratum fixed effects (`Estrato`) and cluster-robust standard errors by primary sampling unit (`Conglomerado`).
- **Complementary OLS-HC3 models and median regression** are unweighted.
- **Confidence intervals for Table 1** use a Taylor linearization approximation based on primary sampling units within strata.


In [5]:
# 4A. Survey estimation functions for Table 1

W = "Fexp_F1F2EX2p_Corr"
STRATA = "Estrato"
PSU = "Conglomerado"


def weighted_quantile(values, quantiles, weights):
    """Compute weighted quantiles using midpoint interpolation."""
    values = np.asarray(values, dtype=float)
    quantiles = np.asarray(quantiles, dtype=float)
    weights = np.asarray(weights, dtype=float)

    ok = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    v = values[ok]
    w = weights[ok]

    if len(v) == 0:
        return np.repeat(np.nan, len(quantiles))

    order = np.argsort(v)
    v, w = v[order], w[order]

    cw = np.cumsum(w) - 0.5 * w
    cw = cw / w.sum()

    return np.interp(quantiles, cw, v)


def survey_mean_ci(data, y, transform=None, alpha=0.05):
    """Estimate a weighted mean and an approximate confidence interval."""
    cols = [y, W, STRATA, PSU]
    z = data[cols].dropna().copy()
    z = z[z[W] > 0]

    x = z[y].astype(float).to_numpy()
    if transform is not None:
        x = transform(x)

    w = z[W].to_numpy(float)

    theta = np.sum(w * x) / np.sum(w)
    z["_lin"] = w * (x - theta) / np.sum(w)

    # Aggregate linearized contributions by PSU within each stratum.
    ps = (
        z.groupby([STRATA, PSU], observed=True)["_lin"]
        .sum()
        .reset_index()
    )

    var = 0.0
    for _, gh in ps.groupby(STRATA, observed=True):
        u = gh["_lin"].to_numpy(float)
        m = len(u)

        if m > 1:
            var += (m / (m - 1.0)) * np.sum((u - u.mean()) ** 2)
        elif m == 1:
            # Retain the original single-PSU stratum adjustment.
            var += float(u[0] ** 2)

    se = math.sqrt(max(var, 0))
    df = max(ps[PSU].nunique() - ps[STRATA].nunique(), 1)
    crit = st.t.ppf(1 - alpha / 2, df)

    lo = theta - crit * se
    hi = theta + crit * se

    return theta, lo, hi, len(z)


def survey_prop_ci(data, mask):
    """Estimate a weighted proportion and its confidence interval."""
    tmp = data.copy()
    tmp["_indicator"] = np.asarray(mask, dtype=float)

    return survey_mean_ci(tmp, "_indicator")


def geometric_mean_ci(data, y):
    """Estimate a weighted geometric mean using positive observations."""
    z = data.loc[data[y] > 0].copy()
    est, lo, hi, n = survey_mean_ci(z, y, transform=np.log)

    return np.exp(est), np.exp(lo), np.exp(hi), n


def fmt_ci(est, lo, hi, pct=False):
    """Format an estimate and its confidence interval."""
    if pct:
        return f"{100 * est:.2f} % ({100 * lo:.2f}–{100 * hi:.2f})"

    return f"{est:.2f} ({lo:.2f}–{hi:.2f})"


def fmt_median(data, col):
    """Format the weighted median and the 25th–75th percentiles."""
    z = data[[col, W]].dropna()
    q = weighted_quantile(z[col], [0.25, 0.50, 0.75], z[W])

    return f"{q[1]:.2f} ({q[0]:.2f}–{q[2]:.2f})"


In [6]:
# 5. Table 1 — Weighted characteristics

rows = []


def add_cont(label, col):
    e, l, u, n = survey_mean_ci(desc, col)
    rows.append({
        "Variable": label,
        "Unweighted n": int(desc[col].notna().sum()),
        "Weighted estimate (95% CI)": fmt_ci(e, l, u),
        "Median (P25–P75)": fmt_median(desc, col),
    })


def add_cat(label, mask, nonmissing_mask):
    base = desc.loc[nonmissing_mask].copy()
    m = mask.loc[base.index]
    e, l, u, n = survey_prop_ci(base, m)

    rows.append({
        "Variable": label,
        "Unweighted n": int(m.sum()),
        "Weighted estimate (95% CI)": fmt_ci(e, l, u, pct=True),
        "Median (P25–P75)": "—",
    })


add_cont("Age, years", "Edad")

sex_valid = desc["Sexo"].notna()
add_cat("Female, %", desc["Sexo"].eq(2), sex_valid)
add_cat("Male, %", desc["Sexo"].eq(1), sex_valid)

# Match the original internal categories and display English labels.
edu_valid = desc["educ_cat"].notna()
for lab, show in [
    ("<8 años", "Education <8 years, %"),
    ("8–12 años", "Education 8–12 years, %"),
    ("≥13 años", "Education ≥13 years, %"),
]:
    add_cat(show, desc["educ_cat"].eq(lab), edu_valid)

sm_valid = desc["tabaco_cat_desc"].notna()
for lab, show in [
    ("Fumador actual", "Current smoker, %"),
    ("Exfumador", "Former smoker, %"),
    ("Nunca fumó", "Never smoker, %"),
]:
    add_cat(show, desc["tabaco_cat_desc"].eq(lab), sm_valid)

al_valid = desc["alcohol_cat_desc"].notna()
for lab, show in [
    ("Nunca/abstemio", "Never drinker/abstainer, %"),
    ("Exbebedor", "Former drinker, %"),
    ("Bebedor actual", "Current drinker, %"),
]:
    add_cat(show, desc["alcohol_cat_desc"].eq(lab), al_valid)

af_valid = desc["af_cat_desc"].notna()
for lab, show in [
    ("Bajo", "Low physical activity, %"),
    ("Medio", "Moderate physical activity, %"),
    ("Alto", "High physical activity, %"),
]:
    add_cat(show, desc["af_cat_desc"].eq(lab), af_valid)

for label, col in [
    ("BMI, kg/m²", "IMC"),
    ("SBP, mmHg", "PAS"),
    ("DBP, mmHg", "PAD"),
    ("Total cholesterol, mg/dL", "Colesterol_Total"),
    ("HDL cholesterol, mg/dL", "Colesterol_HDL"),
    ("Triglycerides, mg/dL", "Trigliceridos"),
    ("Glucose, mg/dL", "Glucosa"),
    ("HbA1c, %", "HbA1c_desc"),
]:
    add_cont(label, col)

# Metal concentrations: weighted geometric means.
for label, col in [
    ("Blood Pb, µg/dL", "Plomo_µgdL"),
    ("Urinary As, µg/L", "Arsenico_µgL"),
    ("Urinary Cd, µg/L", "Cadmio_µgL"),
    ("Urinary Hg, µg/L", "Mercurio_µgL"),
]:
    gm, lo, hi, n = geometric_mean_ci(desc, col)
    med = fmt_median(desc, col) if col == "Plomo_µgdL" else "—"

    rows.append({
        "Variable": label,
        "Unweighted n": int(desc[col].notna().sum()),
        "Weighted estimate (95% CI)": (
            f"GM {gm:.2f} ({lo:.2f}–{hi:.2f})"
        ),
        "Median (P25–P75)": med,
    })

tabla1 = pd.DataFrame(rows)
display(tabla1)


,Variable,Unweighted n,Weighted estimate (95% CI),Median (P25–P75)
0,"Age, years",3847,42.72 (41.73–43.71),41.00 (27.00–56.00)
1,"Female, %",2440,50.88 % (48.29–53.47),—
2,"Male, %",1407,49.12 % (46.53–51.71),—
3,"Education <8 years, %",923,16.71 % (14.72–18.69),—
4,"Education 8–12 years, %",2067,56.60 % (53.25–59.95),—
5,"Education ≥13 years, %",824,26.70 % (23.45–29.95),—
6,"Current smoker, %",1118,34.33 % (31.80–36.85),—
7,"Former smoker, %",904,23.68 % (21.47–25.90),—
8,"Never smoker, %",1825,41.99 % (39.35–44.63),—
9,"Never drinker/abstainer, %",1343,28.75 % (26.24–31.27),—


In [7]:
# 6. Model fitting functions

FULL_COV = (
    "Edad + IMC + C(Zona) + C(Sexo) + C(smoking) + "
    "alcohol12m + edu_years + phys_active"
)


def fit_hc3(outcome):
    """Fit an unweighted OLS model with HC3 robust standard errors."""
    formula = f"{outcome} ~ log2Pb + {FULL_COV}"
    model = smf.ols(formula, data=d).fit(cov_type="HC3")
    lo, hi = model.conf_int().loc["log2Pb"]

    return {
        "n": int(model.nobs),
        "β": model.params["log2Pb"],
        "IC95 % inferior": lo,
        "IC95 % superior": hi,
        "p": model.pvalues["log2Pb"],
        "_model": model,
    }


def fit_weighted(outcome):
    """Fit WLS with stratum fixed effects and cluster-robust covariance."""
    formula = f"{outcome} ~ log2Pb + {FULL_COV} + C(Estrato)"

    cols = [
        outcome, "log2Pb", "Edad", "IMC", "Zona", "Sexo",
        "smoking", "alcohol12m", "edu_years", "phys_active",
        "Estrato", "Conglomerado", W,
    ]

    sub = d[cols].dropna().copy()
    sub = sub[sub[W] > 0]

    model = smf.wls(
        formula,
        data=sub,
        weights=sub[W],
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": sub["Conglomerado"]},
    )

    lo, hi = model.conf_int().loc["log2Pb"]

    return {
        "n": int(model.nobs),
        "β": model.params["log2Pb"],
        "IC95 % inferior": lo,
        "IC95 % superior": hi,
        "p": model.pvalues["log2Pb"],
        "_model": model,
        "_data": sub,
    }


def fit_median(outcome):
    """Fit an unweighted median regression model."""
    model = smf.quantreg(
        f"{outcome} ~ log2Pb + {FULL_COV}",
        data=d,
    ).fit(q=0.5, max_iter=10000)

    lo, hi = model.conf_int().loc["log2Pb"]

    return {
        "n": int(model.nobs),
        "β": model.params["log2Pb"],
        "IC95 % inferior": lo,
        "IC95 % superior": hi,
        "p": model.pvalues["log2Pb"],
        "_model": model,
    }


def clean_result(result):
    """Exclude internal objects from tabular results."""
    return {
        key: value
        for key, value in result.items()
        if not key.startswith("_")
    }

In [8]:
# 7. Table 2 — HC3 models and Benjamini–Hochberg adjustment

outcomes = {
    "SBP, mmHg": "PAS",
    "DBP, mmHg": "PAD",
    "Total cholesterol, mg/dL": "Colesterol_Total",
    "HDL cholesterol, mg/dL": "Colesterol_HDL",
    "Triglycerides, mg/dL": "Trigliceridos",
    "Glucose, mg/dL": "Glucosa",
    "HbA1c, %": "HbA1c_model",
}

hc3_results = {}
rows = []

for label, y in outcomes.items():
    result = fit_hc3(y)
    hc3_results[y] = result
    rows.append({
        "Outcome": label,
        **clean_result(result),
    })

tabla2 = pd.DataFrame(rows).rename(columns={
    "IC95 % inferior": "95% CI lower",
    "IC95 % superior": "95% CI upper",
})

# Adjust p-values for the five secondary metabolic outcomes only.
secondary_names = [
    "Total cholesterol, mg/dL",
    "HDL cholesterol, mg/dL",
    "Triglycerides, mg/dL",
    "Glucose, mg/dL",
    "HbA1c, %",
]

mask = tabla2["Outcome"].isin(secondary_names)
tabla2["BH-adjusted p"] = np.nan

tabla2.loc[mask, "BH-adjusted p"] = multipletests(
    tabla2.loc[mask, "p"].to_numpy(),
    method="fdr_bh",
)[1]

# Round table values after the multiple-testing adjustment.
for col in ["β", "95% CI lower", "95% CI upper"]:
    tabla2[col] = tabla2[col].round(2)

tabla2["p"] = tabla2["p"].round(3)
tabla2["BH-adjusted p"] = tabla2["BH-adjusted p"].round(3)

display(tabla2)


,Outcome,n,β,95% CI lower,95% CI upper,p,BH-adjusted p
0,"SBP, mmHg",3411,0.07,-1.36,1.50,0.925,NaN
1,"DBP, mmHg",3411,0.65,-0.11,1.40,0.095,NaN
2,"Total cholesterol, mg/dL",3406,3.10,0.49,5.70,0.020,0.098
3,"HDL cholesterol, mg/dL",3406,0.78,-0.13,1.69,0.092,0.153
4,"Triglycerides, mg/dL",3406,5.78,-2.10,13.66,0.150,0.188
5,"Glucose, mg/dL",3368,-2.24,-4.47,-0.01,0.049,0.123
6,"HbA1c, %",1229,-0.11,-0.28,0.06,0.213,0.213


In [9]:
# 8. Table 3 — Blood pressure model comparisons

weighted_results = {
    "PAS": fit_weighted("PAS"),
    "PAD": fit_weighted("PAD"),
}

median_results = {
    "PAS": fit_median("PAS"),
    "PAD": fit_median("PAD"),
}

outcome_labels = {
    "PAS": "SBP",
    "PAD": "DBP",
}

rows = []

for y in ["PAS", "PAD"]:
    for method, result in [
        ("Fully adjusted OLS (HC3)", hc3_results[y]),
        ("Weighted analysis (WLS)", weighted_results[y]),
        ("Median regression", median_results[y]),
    ]:
        rows.append({
            "Outcome": outcome_labels[y],
            "Method": method,
            **clean_result(result),
        })

tabla3 = pd.DataFrame(rows).rename(columns={
    "IC95 % inferior": "95% CI lower",
    "IC95 % superior": "95% CI upper",
})

for col in ["β", "95% CI lower", "95% CI upper"]:
    tabla3[col] = tabla3[col].round(2)

tabla3["p"] = tabla3["p"].round(3)

display(tabla3)


,Outcome,Method,n,β,95% CI lower,95% CI upper,p
0,SBP,Fully adjusted OLS (HC3),3411,0.07,-1.36,1.50,0.925
1,SBP,Weighted analysis (WLS),3410,1.16,-0.68,3.01,0.216
2,SBP,Median regression,3411,-0.18,-1.30,0.93,0.746
3,DBP,Fully adjusted OLS (HC3),3411,0.65,-0.11,1.40,0.095
4,DBP,Weighted analysis (WLS),3410,1.15,0.12,2.17,0.028
5,DBP,Median regression,3411,0.76,0.04,1.49,0.038


In [10]:
# 9. Table 3b — Sensitivity analyses for antihypertensive medication use

if ANTIHYP is None or d["antihip"].notna().sum() == 0:
    print(
        "Antihypertensive medication sensitivity analyses were skipped: "
        "no usable medication data are available."
    )
    tabla3b = pd.DataFrame()

else:
    rows = []
    outcome_labels = {"PAS": "SBP", "PAD": "DBP"}

    for y in ["PAS", "PAD"]:
        # Additional adjustment for antihypertensive medication use.
        m_adj = smf.ols(
            f"{y} ~ log2Pb + {FULL_COV} + antihip",
            data=d,
        ).fit(cov_type="HC3")

        lo, hi = m_adj.conf_int().loc["log2Pb"]

        rows.append({
            "Outcome": outcome_labels[y],
            "Analysis": "Additional adjustment for antihypertensive medication use",
            "n": int(m_adj.nobs),
            "β": m_adj.params["log2Pb"],
            "95% CI lower": lo,
            "95% CI upper": hi,
            "p": m_adj.pvalues["log2Pb"],
        })

        # Restrict the sample to participants reporting no medication use.
        sub = d.loc[d["antihip"] == 0].copy()

        m_exc = smf.ols(
            f"{y} ~ log2Pb + {FULL_COV}",
            data=sub,
        ).fit(cov_type="HC3")

        lo, hi = m_exc.conf_int().loc["log2Pb"]

        rows.append({
            "Outcome": outcome_labels[y],
            "Analysis": "Exclusion of antihypertensive medication users",
            "n": int(m_exc.nobs),
            "β": m_exc.params["log2Pb"],
            "95% CI lower": lo,
            "95% CI upper": hi,
            "p": m_exc.pvalues["log2Pb"],
        })

    tabla3b = pd.DataFrame(rows)

    for col in ["β", "95% CI lower", "95% CI upper"]:
        tabla3b[col] = tabla3b[col].round(2)

    tabla3b["p"] = tabla3b["p"].round(3)

    display(tabla3b)


,Outcome,Analysis,n,β,95% CI lower,95% CI upper,p
0,SBP,Additional adjustment for antihypertensive med...,3407,0.34,-1.08,1.76,0.639
1,SBP,Exclusion of antihypertensive medication users,2422,0.35,-1.32,2.02,0.681
2,DBP,Additional adjustment for antihypertensive med...,3407,0.73,-0.03,1.50,0.060
3,DBP,Exclusion of antihypertensive medication users,2422,0.70,-0.23,1.62,0.141


In [11]:
# 10. Cubic splines — Assessment of nonlinearity

from statsmodels.stats.anova import anova_lm
from patsy import bs

spline_rows = []
outcome_labels = {"PAS": "SBP", "PAD": "DBP"}

for y in ["PAS", "PAD"]:
    cols = [
        y, "Edad", "IMC", "Zona", "Sexo", "smoking",
        "alcohol12m", "edu_years", "phys_active", "log2Pb",
    ]
    sub = d[cols].dropna().copy()

    base = f"{y} ~ {FULL_COV}"

    linear = smf.ols(
        base + " + log2Pb",
        data=sub,
    ).fit()

    spline = smf.ols(
        base + " + bs(log2Pb, df=3, degree=3, include_intercept=False)",
        data=sub,
    ).fit()

    # Compare the linear and spline models using an F-test.
    comp = anova_lm(linear, spline)

    spline_rows.append({
        "Desenlace": y,
        "n": int(spline.nobs),
        "p_no_linealidad": comp["Pr(>F)"].iloc[1],
        "AIC_lineal": linear.aic,
        "AIC_spline": spline.aic,
    })

# Retain internal column names for subsequent validation and export.
tabla_splines = pd.DataFrame(spline_rows)

display(
    tabla_splines.assign(
        Desenlace=tabla_splines["Desenlace"].map(outcome_labels)
    )
    .rename(columns={
        "Desenlace": "Outcome",
        "p_no_linealidad": "p for nonlinearity",
        "AIC_lineal": "Linear model AIC",
        "AIC_spline": "Spline model AIC",
    })
    .round(4)
)

,Outcome,n,p for nonlinearity,Linear model AIC,Spline model AIC
0,SBP,3411,0.6849,29032.8899,29036.1299
1,DBP,3411,0.8435,25171.7398,25175.3980


In [12]:
# 11. Table 4 — Interaction analyses

def fit_interaction(outcome, modifier, categorical=False):
    """Fit a weighted interaction model and test interaction terms jointly."""
    term = f"C({modifier})" if categorical else modifier
    formula = (
        f"{outcome} ~ log2Pb*{term} + {FULL_COV} + C(Estrato)"
    )

    needed = [
        outcome, "log2Pb", "Edad", "IMC", "Zona", "Sexo", "smoking",
        "alcohol12m", "edu_years", "phys_active",
        "Estrato", "Conglomerado", W, modifier,
    ]

    # Remove duplicate column names before selecting complete cases.
    sub = d[list(dict.fromkeys(needed))].dropna().copy()
    sub = sub[sub[W] > 0]

    model = smf.wls(
        formula,
        data=sub,
        weights=sub[W],
    ).fit(
        cov_type="cluster",
        cov_kwds={"groups": sub["Conglomerado"]},
    )

    # Identify interaction coefficients for the joint Wald test.
    terms = [
        name for name in model.params.index
        if ":" in name and "log2Pb" in name and modifier in name
    ]

    R = np.zeros((len(terms), len(model.params)))
    for i, name in enumerate(terms):
        R[i, list(model.params.index).index(name)] = 1

    test = model.wald_test(R, scalar=True)

    return float(test.pvalue), model, int(model.nobs)


rows = []

for label, modifier, categorical in [
    ("Pb × age", "Edad", False),
    ("Pb × obesity", "obesidad", False),
    ("Pb × geographic macrozone", "macrozona", True),
]:
    p_dbp, _, _ = fit_interaction("PAD", modifier, categorical)
    p_sbp, _, _ = fit_interaction("PAS", modifier, categorical)

    rows.append({
        "Interaction": label,
        "p for DBP": p_dbp,
        "p for SBP": p_sbp,
    })

tabla4 = pd.DataFrame(rows)
display(tabla4.round(4))

,Interaction,p for DBP,p for SBP
0,Pb × age,0.1682,0.7818
1,Pb × obesity,0.7375,0.9052
2,Pb × geographic macrozone,0.0037,0.0266


In [13]:
# 12. Exploratory linear quantile g-computation
# Four metals; four requested categories; 200 bootstrap resamples; seed 42.
# Tied values may reduce the effective number of exposure categories.
# Bootstrap resampling is performed at the individual level.

METALS = {
    "Pb": "Plomo_µgdL",
    "As": "Arsenico_µgL",
    "Cd": "Cadmio_µgL",
    "Hg": "Mercurio_µgL",
}


def quantize_with_ties(x, q=4):
    """Create quantile-based categories, dropping duplicate bin edges."""
    x = pd.Series(x)
    out = pd.Series(np.nan, index=x.index, dtype=float)
    ok = x.notna()

    out.loc[ok] = pd.qcut(
        x.loc[ok],
        q=q,
        labels=False,
        duplicates="drop",
    )

    return out


def qgcomp_linear(outcome, data, B=200, seed=42):
    """Estimate a joint association using WLS and individual bootstrap."""
    need = [
        outcome, W, "Edad", "IMC", "Zona", "Sexo",
        "smoking", "alcohol12m", "edu_years", "phys_active",
    ] + list(METALS.values())

    z = data[need].dropna().copy()
    z = z[z[W] > 0].copy()

    qcols = []
    cat_info = []

    for metal, col in METALS.items():
        qcol = f"q_{metal}"
        z[qcol] = quantize_with_ties(z[col], q=4)
        qcols.append(qcol)

        cat_info.append({
            "Metal": metal,
            "n": int(z[col].notna().sum()),
            "n_categorías_efectivas": int(z[qcol].nunique()),
            "conteos_por_categoría": str(
                z[qcol].value_counts().sort_index().to_dict()
            ),
        })

    z = z.dropna(subset=qcols).copy()

    # Use the main model covariates, not the descriptive categories.
    formula = (
        f"{outcome} ~ " + " + ".join(qcols)
        + " + Edad + IMC + C(Zona) + C(Sexo) + C(smoking)"
        + " + alcohol12m + edu_years + phys_active"
    )

    model = smf.wls(formula, data=z, weights=z[W]).fit()

    betas = np.array(
        [model.params[col] for col in qcols],
        dtype=float,
    )
    psi = float(betas.sum())

    # Normalize positive and negative coefficients separately.
    pos = np.clip(betas, 0, None)
    neg = np.clip(-betas, 0, None)

    posw = pos / pos.sum() if pos.sum() > 0 else np.zeros_like(pos)
    negw = neg / neg.sum() if neg.sum() > 0 else np.zeros_like(neg)

    # Resample individuals, retaining the original exposure categories.
    rng = np.random.default_rng(seed)
    boot = []
    n = len(z)

    for _ in range(B):
        idx = rng.integers(0, n, n)
        zb = z.iloc[idx].copy()

        try:
            mb = smf.wls(formula, data=zb, weights=zb[W]).fit()
            boot.append(float(sum(mb.params[col] for col in qcols)))
        except Exception:
            # Failed fits are omitted from the effective replicate count.
            pass

    boot = np.asarray(
        [value for value in boot if np.isfinite(value)],
        dtype=float,
    )

    if len(boot) >= 20:
        lo, hi = np.quantile(boot, [0.025, 0.975])
        se = boot.std(ddof=1)
        p_boot = 2 * min(
            np.mean(boot <= 0),
            np.mean(boot >= 0),
        )
    else:
        lo = hi = se = p_boot = np.nan

    # Retain internal column names for subsequent cells.
    summary = pd.DataFrame([{
        "Desenlace": outcome,
        "n": len(z),
        "psi": psi,
        "EE_boot": se,
        "IC95_inf_boot": lo,
        "IC95_sup_boot": hi,
        "p_boot_empírico": p_boot,
        "B_efectivas": len(boot),
    }])

    weights = pd.DataFrame({
        "Metal": list(METALS.keys()),
        "beta_cuantil": betas,
        "contribución_positiva": posw,
        "contribución_negativa": negw,
    })

    return summary, weights, pd.DataFrame(cat_info), z


qg_PAD, w_PAD, cats_PAD, zq_PAD = qgcomp_linear(
    "PAD", d, B=200, seed=42
)
qg_PAS, w_PAS, cats_PAS, zq_PAS = qgcomp_linear(
    "PAS", d, B=200, seed=42
)

tabla_qgcomp = pd.concat([qg_PAD, qg_PAS], ignore_index=True)
tabla_qgcomp_pesos_PAD = w_PAD.copy()
tabla_qgcomp_categorias_PAD = cats_PAD.copy()

# Display English labels without changing the stored tables.
print("Mixture association estimates")
display(
    tabla_qgcomp.assign(
        Desenlace=tabla_qgcomp["Desenlace"].map({
            "PAD": "DBP",
            "PAS": "SBP",
        })
    )
    .rename(columns={
        "Desenlace": "Outcome",
        "psi": "ψ",
        "EE_boot": "Bootstrap SE",
        "IC95_inf_boot": "Bootstrap 95% CI lower",
        "IC95_sup_boot": "Bootstrap 95% CI upper",
        "p_boot_empírico": "Empirical bootstrap p",
        "B_efectivas": "Effective bootstrap replicates",
    })
    .round(4)
)

print("Metal contributions — DBP")
display(
    tabla_qgcomp_pesos_PAD.rename(columns={
        "beta_cuantil": "Category coefficient",
        "contribución_positiva": "Positive contribution",
        "contribución_negativa": "Negative contribution",
    }).round(4)
)

print("Effective exposure categories — DBP")
display(
    tabla_qgcomp_categorias_PAD.rename(columns={
        "n_categorías_efectivas": "Effective categories",
        "conteos_por_categoría": "Counts by category",
    })
)



Mixture association estimates


,Outcome,n,ψ,Bootstrap SE,Bootstrap 95% CI lower,Bootstrap 95% CI upper,Empirical bootstrap p,Effective bootstrap replicates
0,DBP,3063,1.4030,0.6818,-0.0146,2.5898,0.06,200
1,SBP,3063,2.1474,1.1958,0.0698,4.6340,0.05,200


Metal contributions — DBP


,Metal,Category coefficient,Positive contribution,Negative contribution
0,Pb,1.389,0.99,0.0
1,As,0.014,0.01,0.0
2,Cd,0.000,0.00,0.0
3,Hg,-0.000,0.00,1.0


Effective exposure categories — DBP


,Metal,n,Effective categories,Counts by category
0,Pb,3063,2,"{0.0: 2304, 1.0: 759}"
1,As,3063,4,"{0.0: 768, 1.0: 766, 2.0: 763, 3.0: 766}"
2,Cd,3063,1,{0.0: 3063}
3,Hg,3063,1,{0.0: 3063}


In [14]:
# 13. Comparison with manuscript reference values
# Reference values are used after model fitting and do not alter estimates.

reference_core = pd.DataFrame(
    [
        ["PAS", "HC3", 3411, 0.07, -1.36, 1.50, 0.925],
        ["PAD", "HC3", 3411, 0.65, -0.11, 1.40, 0.095],
        ["PAS", "Ponderado", 3410, 1.16, -0.68, 3.01, 0.216],
        ["PAD", "Ponderado", 3410, 1.15, 0.12, 2.17, 0.028],
        ["PAS", "Mediana", 3411, -0.18, -1.30, 0.93, 0.746],
        ["PAD", "Mediana", 3411, 0.76, 0.04, 1.49, 0.038],
    ],
    columns=[
        "Desenlace", "Método", "n_ref",
        "beta_ref", "lo_ref", "hi_ref", "p_ref",
    ],
)

calc = []

for y in ["PAS", "PAD"]:
    for method, result in [
        ("HC3", hc3_results[y]),
        ("Ponderado", weighted_results[y]),
        ("Mediana", median_results[y]),
    ]:
        calc.append([
            y,
            method,
            result["n"],
            result["β"],
            result["IC95 % inferior"],
            result["IC95 % superior"],
            result["p"],
        ])

calc = pd.DataFrame(
    calc,
    columns=[
        "Desenlace", "Método", "n_calc",
        "beta_calc", "lo_calc", "hi_calc", "p_calc",
    ],
)

control = reference_core.merge(
    calc,
    on=["Desenlace", "Método"],
    validate="one_to_one",
)

# Compare sample sizes exactly and estimates at manuscript precision.
control["coincide_redondeo"] = (
    (control["n_ref"] == control["n_calc"])
    & (control["beta_ref"].round(2) == control["beta_calc"].round(2))
    & (control["lo_ref"].round(2) == control["lo_calc"].round(2))
    & (control["hi_ref"].round(2) == control["hi_calc"].round(2))
    & (control["p_ref"].round(3) == control["p_calc"].round(3))
)

# Display English labels while preserving internal names for export.
control_display = control.copy()

control_display["Desenlace"] = control_display["Desenlace"].map({
    "PAS": "SBP",
    "PAD": "DBP",
})

control_display["Método"] = control_display["Método"].map({
    "HC3": "OLS-HC3",
    "Ponderado": "Weighted WLS",
    "Mediana": "Median regression",
})

control_display = control_display.rename(columns={
    "Desenlace": "Outcome",
    "Método": "Method",
    "n_ref": "Reference n",
    "beta_ref": "Reference β",
    "lo_ref": "Reference CI lower",
    "hi_ref": "Reference CI upper",
    "p_ref": "Reference p",
    "n_calc": "Calculated n",
    "beta_calc": "Calculated β",
    "lo_calc": "Calculated CI lower",
    "hi_calc": "Calculated CI upper",
    "p_calc": "Calculated p",
    "coincide_redondeo": "Matches reported precision",
})

display(control_display)

if control["coincide_redondeo"].all():
    print(
        "All six core results match the reference values "
        "at the reported precision."
    )
else:
    print(
        "Discrepancies were detected in the core results. "
        "Review the input data, variable coding, analytical samples, "
        "model specifications, and software versions."
    )

,Outcome,Method,Reference n,Reference β,Reference CI lower,Reference CI upper,Reference p,Calculated n,Calculated β,Calculated CI lower,Calculated CI upper,Calculated p,Matches reported precision
0,SBP,OLS-HC3,3411,0.07,-1.36,1.50,0.925,3411,0.068517,-1.358587,1.495620,0.925030,True
1,DBP,OLS-HC3,3411,0.65,-0.11,1.40,0.095,3411,0.646649,-0.111528,1.404826,0.094593,True
2,SBP,Weighted WLS,3410,1.16,-0.68,3.01,0.216,3410,1.162611,-0.681137,3.006360,0.216498,True
3,DBP,Weighted WLS,3410,1.15,0.12,2.17,0.028,3410,1.145562,0.121830,2.169294,0.028292,True
4,SBP,Median regression,3411,-0.18,-1.30,0.93,0.746,3411,-0.183345,-1.295141,0.928452,0.746465,True
5,DBP,Median regression,3411,0.76,0.04,1.49,0.038,3411,0.764180,0.041427,1.486933,0.038243,True


All six core results match the reference values at the reported precision.


In [15]:
# Software versions for reproducibility

import platform
from importlib.metadata import version

packages = [
    "pandas",
    "numpy",
    "scipy",
    "statsmodels",
    "patsy",
    "pyreadstat",
    "openpyxl",
]

versiones = pd.DataFrame({
    "Component": ["Python"] + packages,
    "Version": [platform.python_version()] + [
        version(package) for package in packages
    ],
})

display(versiones)


,Component,Version
0,Python,3.13.15
1,pandas,2.2.3
2,numpy,2.1.3
3,scipy,1.16.3
4,statsmodels,0.15.0
5,patsy,1.0.3
6,pyreadstat,1.3.6
7,openpyxl,3.1.5


In [16]:
# 14. Export analysis tables and software versions

from pathlib import Path
import pandas as pd
from google.colab import files

OUTDIR = Path("/content/ens_analysis_results")
OUTDIR.mkdir(parents=True, exist_ok=True)


def english_export(table):
    """Translate retained internal labels without modifying the source table."""
    result = table.copy()

    if "Desenlace" in result.columns:
        result["Desenlace"] = result["Desenlace"].replace({
            "PAS": "SBP",
            "PAD": "DBP",
        })

    if "Método" in result.columns:
        result["Método"] = result["Método"].replace({
            "HC3": "OLS-HC3",
            "Ponderado": "Weighted WLS",
            "Mediana": "Median regression",
        })

    return result.rename(columns={
        "Desenlace": "Outcome",
        "Método": "Method",
        "p_no_linealidad": "p for nonlinearity",
        "AIC_lineal": "Linear model AIC",
        "AIC_spline": "Spline model AIC",
        "psi": "ψ",
        "EE_boot": "Bootstrap SE",
        "IC95_inf_boot": "Bootstrap 95% CI lower",
        "IC95_sup_boot": "Bootstrap 95% CI upper",
        "p_boot_empírico": "Empirical bootstrap p",
        "B_efectivas": "Effective bootstrap replicates",
        "beta_cuantil": "Category coefficient",
        "contribución_positiva": "Positive contribution",
        "contribución_negativa": "Negative contribution",
        "n_categorías_efectivas": "Effective categories",
        "conteos_por_categoría": "Counts by category",
        "n_ref": "Reference n",
        "beta_ref": "Reference β",
        "lo_ref": "Reference CI lower",
        "hi_ref": "Reference CI upper",
        "p_ref": "Reference p",
        "n_calc": "Calculated n",
        "beta_calc": "Calculated β",
        "lo_calc": "Calculated CI lower",
        "hi_calc": "Calculated CI upper",
        "p_calc": "Calculated p",
        "coincide_redondeo": "Matches reported precision",
    })


tables = {
    "Table_1_Characteristics": tabla1,
    "Table_2_HC3": tabla2,
    "Table_3_BP_Models": tabla3,
    "Table_3b_Medication": tabla3b,
    "Table_4_Interactions": tabla4,
    "Splines": tabla_splines,
    "QGCOMP_Summary": tabla_qgcomp,
    "QGCOMP_Contributions_DBP": tabla_qgcomp_pesos_PAD,
    "QGCOMP_Categories_DBP": tabla_qgcomp_categorias_PAD,
    "QGCOMP_Contributions_SBP": w_PAS,
    "QGCOMP_Categories_SBP": cats_PAS,
    "Manuscript_Comparison": control,
    "Software_Versions": versiones,
}

export_tables = {}

for name, table in tables.items():
    if isinstance(table, pd.DataFrame) and not table.empty:
        export_tables[name] = english_export(table)
    else:
        print(f"Skipped empty or unavailable table: {name}")

for name, table in export_tables.items():
    table.to_csv(
        OUTDIR / f"{name}.csv",
        index=False,
        encoding="utf-8-sig",
    )

xlsx = OUTDIR / "ENS_Analysis_Tables.xlsx"

with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
    for name, table in export_tables.items():
        table.to_excel(
            writer,
            sheet_name=name[:31],
            index=False,
        )

print(f"Excel workbook created: {xlsx}")
print(f"CSV tables saved to: {OUTDIR}")
print(f"Tables exported: {len(export_tables)}")

files.download(str(xlsx))


Excel workbook created: /content/ens_analysis_results/ENS_Analysis_Tables.xlsx
CSV tables saved to: /content/ens_analysis_results
Tables exported: 13


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Interpreting the Results Check

Review the `Manuscript_Comparison` sheet in the exported workbook.

- `True` in `Matches reported precision` indicates agreement with the manuscript reference at the reported rounding precision.
- `False` indicates a discrepancy. Review the input data, analytical sample, variable coding, model specifications, and software versions before updating the manuscript.
- This check covers six core blood pressure results, not the entire analysis.
- The mixture analysis is exploratory. Tied exposure values may produce fewer than four effective categories; review the category counts before interpreting the estimates.

The exported workbook contains each table in a separate sheet, together with the software versions used.
